# E26 - Change the training TARGET (hazard / ramp / deweight)

**Paradigm lever, not a feature.** Every prior experiment kept the label `1{t >= tau}` and
fit a per-step binary classifier. TS-AUC only cares about the *per-step cross-sectional
ranking*, so a target that encodes *how far past the break we are* (a hazard/ramp) or that
*down-weights the ambiguous points near tau* (E7: near-tau points are the hardest) may
rank better with **no new features**.

Features are frozen to the champion (base15 x (60,130,250) = 45). Only the **target** and
**model head** change. Baseline = champion binary classifier. Candidates:

- `ramp{H}`  - regression target `clip((t-tau)/H, 0, 1)` (0 before break), HGB **regressor**.
- `deweight{H}` - binary target but sample-weight `min(1, |t-tau|/H)` (down-weight near tau).

Hardened bar: only `t >= 2.5` vs champion is worth pursuing. `SMOKE=1` for a quick check.

In [ ]:
import os, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from collections import deque
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

from _overnight import (
    load_pool, select_picks, evaluate, paired, SMOKE, BASE, MIN_N, N_JOBS,
)
from _exp_fast import HIST_CAP, PREDICT_EVERY, hist_context, split_series, ts_auc_from
print("SMOKE =", SMOKE)

from _feat_lib import _feat

BASE_KEYS = ("spec_entropy",)   # base15 builder
SCALES = (60, 130, 250)         # champion multiscale
WINDOW = 250


def _subwin(w, s):
    return w if s >= len(w) else w[-s:]


def champion_feat(w, ctx):
    out = []
    for s in SCALES:
        out.extend(_feat(_subwin(w, s), ctx, keys=BASE_KEYS))
    return out

def verdict(md_, t):
    if md_ > 0 and t >= 2.5:
        return "SHIP(hard)"
    if md_ > 0 and t >= 2.0:
        return "marginal(HOLD)"
    if md_ > 0 and t >= 1.5:
        return "weak"
    return "noise"


In [ ]:
# Custom training/streaming that supports regression targets and sample weights.
def build_training_tgt(X, yi, picks, feat_fn, window, target_fn):
    rows, ys, ws = [], [], []
    for sid, ts in picks.items():
        x_h, onl = split_series(X.loc[sid])
        x_on = onl.to_numpy(dtype=np.float64)
        ctx = hist_context(x_h, HIST_CAP)
        tau = int(yi.loc[sid, "tau_index"])
        for t in ts:
            lo = max(0, t - window + 1)
            rows.append(feat_fn(x_on[lo:t + 1], ctx))
            y, w = target_fn(t, tau)
            ys.append(y)
            ws.append(w)
    return (np.asarray(rows, dtype=np.float64),
            np.asarray(ys, dtype=np.float64),
            np.asarray(ws, dtype=np.float64))


def _stream_score(model, feat_fn, window, sid, x_h, x_on, onl_index, score_fn):
    ctx = hist_context(x_h, HIST_CAP)
    buf = deque()
    out = np.zeros(len(x_on))
    score = 0.0
    for idx, point in enumerate(x_on):
        if len(buf) == window:
            buf.popleft()
        buf.append(float(point))
        if len(buf) < MIN_N:
            out[idx] = 0.0
            continue
        if idx % PREDICT_EVERY == 0:
            feats = np.asarray(feat_fn(np.asarray(buf), ctx), dtype=np.float64).reshape(1, -1)
            score = float(score_fn(model, feats))
        out[idx] = score
    midx = pd.MultiIndex.from_product([[sid], onl_index], names=["id", "time"])
    return pd.Series(out, index=midx)


def evaluate_tgt(feat_fn, window, target_fn, make_model, score_fn,
                 X, yi, y_val, picks, folds):
    Xtr, ytr, wtr = build_training_tgt(X, yi, picks, feat_fn, window, target_fn)
    model = make_model()
    with threadpool_limits(limits=1):
        model.fit(Xtr, ytr, sample_weight=wtr)
    aucs = []
    for f in folds:
        jobs = []
        for sid in f:
            x_h, onl = split_series(X.loc[sid])
            jobs.append((sid, x_h, onl.to_numpy(dtype=np.float64), onl.index))
        with threadpool_limits(limits=1):
            series = Parallel(n_jobs=N_JOBS, backend="loky")(
                delayed(_stream_score)(model, feat_fn, window, sid, x_h, x_on, oi, score_fn)
                for (sid, x_h, x_on, oi) in jobs
            )
        aucs.append(ts_auc_from(pd.concat(series).to_frame("prediction"), y_val))
    return np.asarray(aucs)


# target builders
def make_ramp(H):
    def f(t, tau):
        if tau < 0:
            return 0.0, 1.0
        return float(min(max((t - tau) / H, 0.0), 1.0)), 1.0
    return f


def make_deweight(H):
    def f(t, tau):
        y = 1.0 if (tau >= 0 and t >= tau) else 0.0
        w = 1.0 if tau < 0 else float(min(1.0, abs(t - tau) / H))
        return y, w
    return f


_clf = lambda: HistGradientBoostingClassifier(**BASE)
_reg = lambda: HistGradientBoostingRegressor(**BASE)
_proba = lambda m, f: m.predict_proba(f)[0, 1]
_pred = lambda m, f: m.predict(f)[0]


In [ ]:
X, yi, y_val, train_ids, folds = load_pool()
picks = select_picks(X, train_ids)
print(f"pool {len(train_ids)} train, {len(folds)} folds x {len(folds[0])}")

base_aucs, nf = evaluate(champion_feat, WINDOW, X, yi, y_val, picks, folds)
print(f"champion   nf={nf} mean={base_aucs.mean():.4f}")

H = 10 if SMOKE else 20
candidates = {
    f"ramp{H}":     (make_ramp(H),     _reg, _pred),
    f"deweight{H}": (make_deweight(H), _clf, _proba),
    f"ramp{H*2}":   (make_ramp(H * 2), _reg, _pred),
}
for name, (tfn, mk, sfn) in candidates.items():
    a = evaluate_tgt(champion_feat, WINDOW, tfn, mk, sfn, X, yi, y_val, picks, folds)
    md_, sd, t, _ = paired(a, base_aucs)
    print(f"{name:>10}  mean={a.mean():.4f}  delta={md_:+.4f}  t={t:+.2f}  -> {verdict(md_, t)}")


## Interpretation

If any `ramp`/`deweight` variant clears **t >= 2.5**, the win comes for free (same features,
just a better target) - port the target+head into the champion notebook and local-test.
If all are noise, the binary target is already optimal for this metric and we move on.